# Imports

In [1]:
import os
import pickle
import re
import shutil
import sys
sys.path.append(os.path.dirname(os.getcwd()))
from itertools import product

import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import yaml
from matplotlib.backends.backend_pdf import PdfPages
from tools import load_npy, load_yaml_as_df, load_pkl, exist_metric, exist_stf_metric, inverse_stf_metrics, keep_split, is_full_group

plt.style.use('default')
matplotlib.rcParams['pdf.fonttype'] = 42
matplotlib.rcParams['ps.fonttype'] = 42
plt.rc('font', family='Arial')
matplotlib.rcParams['mathtext.fontset'] = 'stix'
matplotlib.rcParams['font.size'] = 10

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

/usr/local/lib/python3.10/dist-packages/cupy/_environment.py:596: UserWarning: 
--------------------------------------------------------------------------------

  CuPy may not function correctly because multiple CuPy packages are installed
  in your environment:

    cupy-cuda11x, cupy-cuda12x

  Follow these steps to resolve this issue:

    1. For all packages listed above, run the following command to remove all
       existing CuPy installations:

         $ pip uninstall <package_name>

      If you previously installed CuPy via conda, also run the following:

         $ conda uninstall cupy

    2. Install the appropriate CuPy package.
       Refer to the Installation Guide for detailed instructions.

         https://docs.cupy.dev/en/stable/install.html

--------------------------------------------------------------------------------

  warnings.warn(f'''


[1758620752.876754] [qs-39567-851088-ai-122756-default0-0:3785904:f]        vfs_fuse.c:281  UCX  ERROR inotify_add_watch(/tmp) failed: No space left on device


# load data

In [ ]:
root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/results_ML3/finetune'
exp_dirs = os.listdir(root)
exp_dirs = [os.path.join(root, exp_dir) for exp_dir in exp_dirs]

params = ['model', 'pred_len', 'data_id', 'learning_rate', 'inner_lr', 'meta_lr', 'rec_lambda', 'auxi_lambda', 'reg_lambda', 'lradj', 'train_epochs', 'patience', 'batch_size', 'auxi_batch_size', 'warmup_steps', 'meta_inner_steps', 'overlap_ratio', 'num_tasks', 'max_norm', 'auxi_loss', 'first_order', 'dropout', 'cycle']
metric_names = ['mse', 'mae', 'cov']

df = []
for exp_dir in exp_dirs:
    if 'MAE' in exp_dir:
        continue
    # if 'Traffic' not in exp_dir and 'ECL' not in exp_dir and 'Weather' not in exp_dir:
    #     continue
    # if 'Traffic' not in exp_dir or 'TQNet' not in exp_dir:
    #     continue
    runned, setting_dir = exist_metric(exp_dir)
    if not runned:
        continue

    config = load_yaml_as_df(os.path.join(setting_dir, 'config.yaml'))
    metric = load_npy(os.path.join(setting_dir, 'metrics.npy'))
    result = config[params]
    if len(metric) == 6:
        log_metrics = load_metric_from_log(os.path.join(exp_dir, 'result_long_term_forecast.txt'))
        cov_loss = log_metrics['cov']
        result.loc[:, metric_names] = metric[1], metric[0], cov_loss
    else:
        result.loc[:, metric_names] = metric[1], metric[0], metric[2]
    result.loc[:, ['meta_type']] = config[['meta_type']] if 'meta_type' in config.columns else 'all'
    result.loc[:, ['exp_dir']] = exp_dir
    df.append(result)

df = pd.concat(df, ignore_index=True)

df.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)

save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
os.makedirs(save_root, exist_ok=True)
df.to_csv(f"{save_root}/finetune_all_results.csv", index=False)

df.head(4)

## pre-load

In [20]:
save_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
df = pd.read_csv(f'{save_root}/finetune_all_results.csv')

df.head(4)

,model,pred_len,data_id,learning_rate,inner_lr,meta_lr,rec_lambda,auxi_lambda,reg_lambda,lradj,train_epochs,patience,batch_size,auxi_batch_size,warmup_steps,meta_inner_steps,overlap_ratio,num_tasks,max_norm,auxi_loss,first_order,dropout,cycle,mse,mae,cov,meta_type,exp_dir
0,Fredformer,96,ECL,0.002,0.002,0.05,1.0,0.0,0.0,type1,100,5,32,64,300,1,0.0,3,5.0,MSE,1,0.2,24,0.154554,0.245372,0.103599,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
1,Fredformer,96,ECL,0.002,0.002,0.05,1.0,0.0,0.0,type1,100,5,32,64,300,2,0.0,3,5.0,MSE,1,0.2,24,0.153540,0.244694,0.102987,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
2,Fredformer,96,ECL,0.002,0.002,0.10,1.0,0.0,0.0,type1,100,5,32,64,300,1,0.0,3,5.0,MSE,1,0.2,24,0.152893,0.244035,0.087282,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
3,Fredformer,96,ECL,0.002,0.002,0.10,1.0,0.0,0.0,type1,100,5,32,64,300,2,0.0,3,5.0,MSE,1,0.2,24,0.153492,0.244443,0.087609,all,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...


## preprocess

In [21]:
stats_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
log_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/logs'
baselines = pd.read_csv(f'{log_root}/baselines_chosen.csv')
best = pd.read_csv(f'{stats_root}/long_term.csv')

datasets = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather']
base = baselines.copy()
base = base[
    (base.data_id.isin(datasets) & (base.model == 'TQNet'))
]
base['model'] = 'DF'

aba1 = df.copy()
aba1 = aba1[aba1.meta_type == 'diag']
aba1 = aba1[(aba1.model == 'TQNet') & (aba1.data_id.isin(datasets))]
aba1['model'] = r'MetaDF$^\dagger$'

aba2 = df.copy()
aba2 = aba2[aba2.meta_type == 'off_diag']
aba2 = aba2[(aba2.model == 'TQNet') & (aba2.data_id.isin(datasets))]
aba2['model'] = r'MetaDF$^\ddagger$'

aba3 = best.copy()
aba3 = aba3[aba3.data_id.isin(datasets)]


In [7]:
print(len(aba1.data_id.unique()), aba1.data_id.unique())
print(len(aba2.data_id.unique()), aba2.data_id.unique())
print(len(aba3.data_id.unique()), aba3.data_id.unique())


6 ['ECL' 'ETTh1' 'ETTh2' 'ETTm1' 'ETTm2' 'Weather']
6 ['ECL' 'ETTh1' 'ETTh2' 'ETTm1' 'ETTm2' 'Weather']
6 ['ETTm1' 'ETTm2' 'ETTh1' 'ETTh2' 'ECL' 'Weather']


## analysis base

In [22]:
df1 = base.copy()

df1 = df1[['model', 'pred_len', 'data_id', 'mse', 'mae']]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather']
df1['data_id'] = pd.Categorical(df1['data_id'], categories=dst_order, ordered=True)

df1_avg = df1.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df1_avg['pred_len'] = 'Avg'
df1 = pd.concat([df1, df1_avg]).reset_index(drop=True)

df1.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)
df1.reset_index(drop=True, inplace=True)

df1.dropna(inplace=True, thresh=4)
df1.head(5)

/tmp/ipykernel_3785904/1556846757.py:8: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df1_avg = df1.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae
0,DF,96,ETTm1,0.310382,0.351721
1,DF,192,ETTm1,0.356100,0.377443
2,DF,336,ETTm1,0.387569,0.399900
3,DF,720,ETTm1,0.450021,0.436546
4,DF,Avg,ETTm1,0.376018,0.391403


## analysis ablation 1

In [18]:
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'learning_rate', 'rec_lambda', 'auxi_lambda', 'lradj', 'train_epochs', 'patience', 'mask_factor', 'distance', 'joint_forecast', 'ot_type', 'batch_size', 'auxi_loss', 'reg_sk', 'eps', r'$w_\mu$', r'$w_\sigma$']
aba1[(aba1['pred_len'] == 192) & (aba1['data_id'] == 'ETTm1')].sort_values(by=['pred_len', 'mse'])[columns].head(10)
aba1[(aba1['pred_len'] == 96) & (aba1['data_id'] == 'ETTh1')].sort_values(by=['pred_len', 'mse'])[columns].head(20)
# aba1[(aba1['pred_len'] == 336) & (aba1['data_id'] == 'ECL')].sort_values(by=['pred_len', 'mse'])[columns].head(10)
# aba1[(aba1['pred_len'] == 720) & (aba1['data_id'] == 'ETTm2')].sort_values(by=['pred_len', 'mse'])[columns].head(10)
# aba1[(aba1['pred_len'] == 720) & (aba1['data_id'] == 'ETTh2')].sort_values(by=['pred_len', 'mse'])[columns].head(10)
# aba1[(aba1['pred_len'] == 720) & (aba1['data_id'] == 'Traffic')].sort_values(by=['pred_len', 'mse'])[columns].head(10)
# aba1[(aba1['pred_len'] == 12) & (aba1['data_id'] == 'PEMS08')].sort_values(by=['pred_len', 'mse'])[columns].head(10)

,model,pred_len,data_id,mse,mae,learning_rate,rec_lambda,auxi_lambda,lradj,train_epochs,patience,mask_factor,distance,joint_forecast,ot_type,batch_size,auxi_loss,reg_sk,eps,$w_\mu$,$w_\sigma$
305,DistDF$^\dagger$,96,ETTh1,0.374495,0.392901,0.0001,0.90,0.10,type3,100,10,0.0,wasserstein_empirical_per_dim,1,upper_bound,128,NaN,0.005,1.000000e-09,1.0,0.0
233,DistDF$^\dagger$,96,ETTh1,0.375333,0.393614,0.0001,0.98,0.02,type3,100,10,0.0,wasserstein_empirical_per_dim,1,upper_bound,128,NaN,0.005,1.000000e-09,1.0,0.0
323,DistDF$^\dagger$,96,ETTh1,0.375476,0.394535,0.0003,0.90,0.10,type3,100,10,0.0,wasserstein_empirical_per_dim,1,upper_bound,128,NaN,0.005,1.000000e-09,1.0,0.0
35,DistDF$^\dagger$,96,ETTh1,0.376117,0.393163,0.0003,0.60,0.40,type3,100,10,0.0,wasserstein_empirical_per_dim,1,upper_bound,128,NaN,0.005,1.000000e-09,1.0,0.0
107,DistDF$^\dagger$,96,ETTh1,0.376180,0.394141,0.0003,0.70,0.30,type3,100,10,0.0,wasserstein_empirical_per_dim,1,upper_bound,128,NaN,0.005,1.000000e-09,1.0,0.0
161,DistDF$^\dagger$,96,ETTh1,0.376967,0.393398,0.0001,0.80,0.20,type3,100,10,0.0,wasserstein_empirical_per_dim,1,upper_bound,128,NaN,0.005,1.000000e-09,1.0,0.0
251,DistDF$^\dagger$,96,ETTh1,0.377030,0.393753,0.0003,0.98,0.02,type3,100,10,0.0,wasserstein_empirical_per_dim,1,upper_bound,128,NaN,0.005,1.000000e-09,1.0,0.0
341,DistDF$^\dagger$,96,ETTh1,0.378352,0.397262,0.0005,0.90,0.10,type3,100,10,0.0,wasserstein_empirical_per_dim,1,upper_bound,128,NaN,0.005,1.000000e-09,1.0,0.0
179,DistDF$^\dagger$,96,ETTh1,0.378521,0.394733,0.0003,0.80,0.20,type3,100,10,0.0,wasserstein_empirical_per_dim,1,upper_bound,128,NaN,0.005,1.000000e-09,1.0,0.0
197,DistDF$^\dagger$,96,ETTh1,0.379449,0.396512,0.0005,0.80,0.20,type3,100,10,0.0,wasserstein_empirical_per_dim,1,upper_bound,128,NaN,0.005,1.000000e-09,1.0,0.0


In [ ]:
df2 = aba1.copy()
df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'Traffic') & (df2.learning_rate == 0.001) & (df2.alpha == 0.6)]

In [23]:
min_mode = 'each'

df2 = aba1.copy()
# df2_m1_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ETTm1') & (df2.learning_rate == 0.0005) & (df2.auxi_lambda == 0.05)]
# df2_m1_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ETTm1') & (df2.learning_rate == 0.0002) & (df2.auxi_lambda == 0.05)]
# df2_m1_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'ETTm1') & (df2.learning_rate == 0.0001) & (df2.auxi_lambda == 0.1)]
# df2_m1_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'ETTm1') & (df2.learning_rate == 0.0002) & (df2.auxi_lambda == 0.1)]

# df2_m2_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ETTm2') & (df2.learning_rate == 0.0001) & (df2.alpha == 1.0)]
# df2_m2_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ETTm2') & (df2.learning_rate == 0.0001) & (df2.alpha == 1.0)]
# df2_m2_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'ETTm2') & (df2.learning_rate == 0.0005) & (df2.alpha == 1.0)]
# df2_m2_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'ETTm2') & (df2.learning_rate == 0.0005) & (df2.alpha == 0.9)]

# df2_h1_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ETTh1') & (df2.learning_rate == 0.0001) & (df2.auxi_lambda == 0.02)]
# df2_h1_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ETTh1') & (df2.learning_rate == 0.0001) & (df2.auxi_lambda == 0.2)]
# df2_h1_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'ETTh1') & (df2.learning_rate == 0.0005) & (df2.alpha == 0.9)]
# df2_h1_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'ETTh1') & (df2.learning_rate == 0.0003) & (df2.auxi_lambda == 0.1)]

# df2_h2_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ETTh2') & (df2.learning_rate == 0.0005) & (df2.alpha == 0.9)]
# df2_h2_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'ETTh2') & (df2.learning_rate == 0.00005) & (df2.alpha == 1.0)]

# df2_ecl_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'ECL') & (df2.learning_rate == 0.001) & (df2.alpha == 0.1)]
# df2_ecl_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'ECL') & (df2.learning_rate == 0.001) & (df2.alpha == 0.5)]
# df2_ecl_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'ECL') & (df2.learning_rate == 0.001) & (df2.alpha == 0.7)]

# df2_tra_96 = df2[(df2['pred_len'] == 96) & (df2['data_id'] == 'Traffic') & (df2.learning_rate == 0.001) & (df2.alpha == 0.6) & (df2.batch_size == 8)]
# df2_tra_192 = df2[(df2['pred_len'] == 192) & (df2['data_id'] == 'Traffic') & (df2.learning_rate == 0.001) & (df2.alpha == 0.6) & (df2.batch_size == 8)]
# df2_tra_336 = df2[(df2['pred_len'] == 336) & (df2['data_id'] == 'Traffic') & (df2.learning_rate == 0.001) & (df2.alpha == 0.6) & (df2.batch_size == 8)]
# df2_tra_720 = df2[(df2['pred_len'] == 720) & (df2['data_id'] == 'Traffic') & (df2.learning_rate == 0.001) & (df2.alpha == 0.2) & (df2.batch_size == 8)]

# df2_p8_12 = df2[(df2['pred_len'] == 12) & (df2['data_id'] == 'PEMS08') & (df2.learning_rate == 0.001) & (df2.alpha == 1.0)]

# df2_other = df2[
#     df2.data_id.isin(['ETTm2', 'ETTh2', 'ECL', 'Traffic', 'Weather']) |
#     ((df2.data_id == 'ETTh1') & df2.pred_len.isin([336]))
# ]
# df2 = pd.concat([
#     df2_m1_96, df2_m1_192, df2_m1_336, df2_m1_720,
#     df2_h1_96, df2_h1_192, df2_h1_720,
#     df2_other
# ], ignore_index=True)


min_mse_idx = df2.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
df2 = df2.loc[min_mse_idx]

df2 = df2[['model', 'pred_len', 'data_id', 'mse', 'mae']]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather']
df2['data_id'] = pd.Categorical(df2['data_id'], categories=dst_order, ordered=True)

df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df2_avg['pred_len'] = 'Avg'

df2 = pd.concat([df2, df2_avg]).reset_index(drop=True)
df2.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)
df2.reset_index(drop=True, inplace=True)

df2.dropna(inplace=True, thresh=5)
df2

/tmp/ipykernel_3785904/3954041845.py:52: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df2_avg = df2.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae
0,MetaDF$^\dagger$,96,ETTm1,0.309009,0.350826
1,MetaDF$^\dagger$,192,ETTm1,0.353675,0.377610
2,MetaDF$^\dagger$,336,ETTm1,0.387283,0.401348
3,MetaDF$^\dagger$,720,ETTm1,0.449830,0.439005
4,MetaDF$^\dagger$,Avg,ETTm1,0.374949,0.392197
5,MetaDF$^\dagger$,96,ETTm2,0.171300,0.253905
6,MetaDF$^\dagger$,192,ETTm2,0.235271,0.295499
7,MetaDF$^\dagger$,336,ETTm2,0.292161,0.333783
8,MetaDF$^\dagger$,720,ETTm2,0.391324,0.392267
9,MetaDF$^\dagger$,Avg,ETTm2,0.272514,0.318864


## analysis ablation 2

In [37]:
columns = ['model', 'pred_len', 'data_id', 'mse', 'mae', 'cov', 'learning_rate', 'inner_lr', 'meta_lr', 'auxi_loss', 'rec_lambda', 'auxi_lambda', 'reg_lambda', 'lradj', 'train_epochs', 'patience', 'batch_size', 'auxi_batch_size', 'warmup_steps', 'meta_inner_steps', 'overlap_ratio', 'num_tasks', 'max_norm', 'first_order', 'dropout', 'cycle', 'meta_type', 'exp_dir']
# aba2[(aba2['pred_len'] == 96) & (aba2['data_id'] == 'ETTm1')].sort_values(by=['pred_len', 'mse'])[columns]
# aba2[(aba2['pred_len'] == 336) & (aba2['data_id'] == 'Weather')].sort_values(by=['pred_len', 'mse'])[columns]
aba2[(aba2['pred_len'] == 720) & (aba2['data_id'] == 'ETTh1')].sort_values(by=['pred_len', 'mse'])[columns].head(20)
# aba2[(aba2['pred_len'] == 96) & (aba2['data_id'] == 'ETTh2')].sort_values(by=['pred_len', 'mse'])[columns]
aba2[(aba2['pred_len'] == 336) & (aba2['data_id'] == 'ECL')].sort_values(by=['pred_len', 'mse'])[columns]
# aba2[(aba2['pred_len'] == 96) & (aba2['data_id'] == 'Weather')].sort_values(by=['pred_len', 'mse'])[columns]
# aba2[(aba2['pred_len'] == 48) & (aba2['data_id'] == 'PEMS03')].sort_values(by=['pred_len', 'mse'])[columns]
# aba2[(aba2['pred_len'] == 192) & (aba2['data_id'] == 'Traffic')].sort_values(by=['pred_len', 'mse'])[columns]

,model,pred_len,data_id,mse,mae,cov,learning_rate,inner_lr,meta_lr,auxi_loss,rec_lambda,auxi_lambda,reg_lambda,lradj,train_epochs,patience,batch_size,auxi_batch_size,warmup_steps,meta_inner_steps,overlap_ratio,num_tasks,max_norm,first_order,dropout,cycle,meta_type,exp_dir
1077,MetaDF$^\ddagger$,336,ECL,0.169199,0.261992,0.124892,0.005,0.005,0.10,MSE,1.0,0.0,0.0,type1,30,5,16,64,300,1,0.0,3,5.0,1,0.0,168,off_diag,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
1011,MetaDF$^\ddagger$,336,ECL,0.169239,0.262467,0.147931,0.005,0.005,0.02,MSE,1.0,0.0,0.0,type1,30,5,16,64,300,1,0.0,3,5.0,1,0.0,168,off_diag,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
1044,MetaDF$^\ddagger$,336,ECL,0.169280,0.262433,0.135305,0.005,0.005,0.05,MSE,1.0,0.0,0.0,type1,30,5,16,64,300,1,0.0,3,5.0,1,0.0,168,off_diag,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
978,MetaDF$^\ddagger$,336,ECL,0.170686,0.263712,0.125940,0.002,0.002,0.10,MSE,1.0,0.0,0.0,type1,30,5,16,64,300,1,0.0,3,5.0,1,0.0,168,off_diag,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
970,MetaDF$^\ddagger$,336,ECL,0.170834,0.264004,0.136524,0.002,0.002,0.05,MSE,1.0,0.0,0.0,type1,30,5,16,64,300,1,0.0,3,5.0,1,0.0,168,off_diag,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
962,MetaDF$^\ddagger$,336,ECL,0.171023,0.264227,0.149447,0.002,0.002,0.02,MSE,1.0,0.0,0.0,type1,30,5,16,64,300,1,0.0,3,5.0,1,0.0,168,off_diag,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
1135,MetaDF$^\ddagger$,336,ECL,0.173356,0.266081,0.151532,0.010,0.010,0.02,MSE,1.0,0.0,0.0,type1,30,5,16,64,300,1,0.0,3,5.0,1,0.0,168,off_diag,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
1151,MetaDF$^\ddagger$,336,ECL,0.173508,0.265831,0.127931,0.010,0.010,0.10,MSE,1.0,0.0,0.0,type1,30,5,16,64,300,1,0.0,3,5.0,1,0.0,168,off_diag,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...
1143,MetaDF$^\ddagger$,336,ECL,0.174528,0.266906,0.139456,0.010,0.010,0.05,MSE,1.0,0.0,0.0,type1,30,5,16,64,300,1,0.0,3,5.0,1,0.0,168,off_diag,/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/P...


In [42]:

min_mode = 'each'

df3 = aba2.copy()
# df3_m1_96 = df3[(df3['pred_len'] == 96) & (df3['data_id'] == 'ETTm1') & (df3.rank_ratio == 0.4)]

# df3_h1_96 = df3[(df3['pred_len'] == 96) & (df3['data_id'] == 'ETTh1') & (df3.learning_rate == 0.0005) & (df3.auxi_lambda == 0.4)]
df3_h1_192 = df3[(df3['pred_len'] == 192) & (df3['data_id'] == 'ETTh1') & (df3.learning_rate == 0.0005) & (df3.meta_lr == 0.05)]
df3_h1_336 = df3[(df3['pred_len'] == 336) & (df3['data_id'] == 'ETTh1') & (df3.learning_rate == 0.0005) & (df3.meta_lr == 0.05)]
df3_h1_720 = df3[(df3['pred_len'] == 720) & (df3['data_id'] == 'ETTh1') & (df3.learning_rate == 0.0005) & (df3.meta_lr == 0.2)]

# df3_tra_96 = df3[(df3['pred_len'] == 96) & (df3['data_id'] == 'Traffic') & (df3.rank_ratio == 0.2)]
# df3_tra_192 = df3[(df3['pred_len'] == 192) & (df3['data_id'] == 'Traffic') & (df3.rank_ratio == 0.2)]
# df3_tra_other = df3[((df3['data_id'] == 'Traffic') & (df3['pred_len'].isin([336, 720])))]

df3_ecl_96 = df3[(df3['pred_len'] == 96) & (df3['data_id'] == 'ECL') & (df3.learning_rate == 0.002) & (df3.meta_lr == 0.1)]
# df3_ecl_192 = df3[(df3['pred_len'] == 192) & (df3['data_id'] == 'ECL') & (df3.learning_rate == 0.0005) & (df3.auxi_lambda == 0.03)]
df3_ecl_336 = df3[(df3['pred_len'] == 336) & (df3['data_id'] == 'ECL') & (df3.learning_rate == 0.002) & (df3.meta_lr == 0.1)]
# df3_ecl_720 = df3[(df3['pred_len'] == 720) & (df3['data_id'] == 'ECL') & (df3.learning_rate == 0.0005) & (df3.auxi_lambda == 0.02)]

# df3_wea_96 = df3[(df3['pred_len'] == 96) & (df3['data_id'] == 'Weather') & (df3.learning_rate == 0.0005) & (df3.auxi_lambda == 0.01)]
# df3_wea_192 = df3[(df3['pred_len'] == 192) & (df3['data_id'] == 'Weather') & (df3.rank_ratio == 0.4)]
# df3_wea_336 = df3[(df3['pred_len'] == 336) & (df3['data_id'] == 'Weather') & (df3.rank_ratio == 0.4)]

# df3_p3 = df3[(df3['data_id'] == 'PEMS03') & (df3.learning_rate == 0.0005) & (df3.rank_ratio == 0.4)]

df3_other = df3[
    ((df3['data_id'] == 'ETTh1') & (df3['pred_len'].isin([96]))) |
    ((df3['data_id'] == 'ECL') & (df3['pred_len'].isin([192, 720]))) |
    df3['data_id'].isin(['ETTm1', 'ETTm2', 'ETTh2', 'Weather'])
]

df3 = pd.concat([
    df3_h1_192, df3_h1_336, df3_h1_720,
    df3_ecl_96, df3_ecl_336,
    df3_other
], ignore_index=True)

min_mse_idx = df3.groupby(['model', 'data_id', 'pred_len'])['mse'].idxmin()
df3 = df3.loc[min_mse_idx]

df3 = df3[['model', 'pred_len', 'data_id', 'mse', 'mae']]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather']
df3['data_id'] = pd.Categorical(df3['data_id'], categories=dst_order, ordered=True)

df3_avg = df3.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df3_avg['pred_len'] = 'Avg'

df3 = pd.concat([df3, df3_avg]).reset_index(drop=True)
df3.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)
df3.reset_index(drop=True, inplace=True)

df3.dropna(inplace=True, thresh=5)
df3

/tmp/ipykernel_3785904/3513892116.py:46: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df3_avg = df3.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae
0,MetaDF$^\ddagger$,96,ETTm1,0.308121,0.350558
1,MetaDF$^\ddagger$,192,ETTm1,0.353449,0.376972
2,MetaDF$^\ddagger$,336,ETTm1,0.384804,0.399481
3,MetaDF$^\ddagger$,720,ETTm1,0.443464,0.436364
4,MetaDF$^\ddagger$,Avg,ETTm1,0.372459,0.390844
5,MetaDF$^\ddagger$,96,ETTm2,0.170945,0.253257
6,MetaDF$^\ddagger$,192,ETTm2,0.234660,0.294528
7,MetaDF$^\ddagger$,336,ETTm2,0.290707,0.331587
8,MetaDF$^\ddagger$,720,ETTm2,0.387285,0.389059
9,MetaDF$^\ddagger$,Avg,ETTm2,0.270899,0.317108


## analysis ablation 3

In [25]:
min_mode = 'each'

df4 = aba3.copy()

df4 = df4[['model', 'pred_len', 'data_id', 'mse', 'mae']]

dst_order = ['ETTm1', 'ETTm2', 'ETTh1', 'ETTh2', 'ECL', 'Weather']
df4['data_id'] = pd.Categorical(df4['data_id'], categories=dst_order, ordered=True)

df4_avg = df4.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()
df4_avg['pred_len'] = 'Avg'

df4 = pd.concat([df4, df4_avg]).reset_index(drop=True)
df4.sort_values(by=['data_id', 'model', 'pred_len'], inplace=True)
df4.reset_index(drop=True, inplace=True)

df4.dropna(inplace=True, thresh=5)
df4

/tmp/ipykernel_3785904/3725225895.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df4_avg = df4.groupby(['model', 'data_id']).mean(numeric_only=True).reset_index()


,model,pred_len,data_id,mse,mae
0,MetaDF,96,ETTm1,0.306602,0.348971
1,MetaDF,192,ETTm1,0.352414,0.376268
2,MetaDF,336,ETTm1,0.382595,0.397512
3,MetaDF,720,ETTm1,0.441163,0.434478
4,MetaDF,Avg,ETTm1,0.370693,0.389307
5,MetaDF,96,ETTm2,0.170291,0.252600
6,MetaDF,192,ETTm2,0.233855,0.293836
7,MetaDF,336,ETTm2,0.290130,0.331435
8,MetaDF,720,ETTm2,0.386822,0.388532
9,MetaDF,Avg,ETTm2,0.270274,0.316601


## concat analysis

In [43]:
aba_res = pd.concat([df1, df2, df3, df4], axis=0)
stats_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
# aba_res.round(3)[['data_id', 'pred_len', 'mse', 'mae', 'label']].to_csv(f'{save_root}/aba_res2.csv', index=False, float_format='%.3f')
# aba_res.replace({'pred_len': {12: 96, 24: 192, 36: 336, 48: 720}}, inplace=True)


dst_order = ['ETTm1', 'ETTh1', 'ECL', 'Weather']
res1 = aba_res[(aba_res['data_id'].isin(dst_order))].copy()
res1['data_id'] = pd.Categorical(res1['data_id'], categories=dst_order, ordered=True)

model_order = ['DF', r'MetaDF$^\dagger$', r'MetaDF$^\ddagger$', 'MetaDF']
res1['model'] = pd.Categorical(res1['model'], categories=model_order, ordered=True)

res1.sort_values(by=['model', 'data_id', 'pred_len'], inplace=True)
res1.to_csv(f'{stats_root}/ablation.csv', index=False)

res1 = res1.set_index(['model', 'data_id', 'pred_len']).unstack('pred_len').swaplevel(axis=1)
columns = []
for pl in res1.columns.levels[0]:
    columns.append((pl, 'mse'))
    columns.append((pl, 'mae'))
res1 = res1[columns]

# res1.round(3).to_csv(f'{stats_root}/aba_res_row2_full.csv', float_format='%.3f')

# res1.round(3).to_latex(f'{save_root}/aba_res_row2_full.tex', float_format='%.3f', escape=False, index=True, header=True)


res1


pred_len                         96                 192                 336  \
                                mse       mae       mse       mae       mse   
model             data_id                                                     
DF                ETTm1    0.310382  0.351721  0.356100  0.377443  0.387569   
                  ETTh1    0.372007  0.391440  0.430498  0.424173  0.486262   
                  ECL      0.143456  0.237354  0.160641  0.252081  0.178238   
                  Weather  0.159965  0.202723  0.209872  0.247166  0.266696   
MetaDF$^\dagger$  ETTm1    0.309009  0.350826  0.353675  0.377610  0.387283   
                  ETTh1    0.371988  0.393626  0.432427  0.423812  0.474530   
                  ECL      0.135195  0.229528  0.153531  0.245538  0.169857   
                  Weather  0.158555  0.201612  0.207597  0.245784  0.264595   
MetaDF$^\ddagger$ ETTm1    0.308121  0.350558  0.353449  0.376972  0.384804   
                  ETTh1    0.369035  0.390616  0.429707  0.421995  0.477403   
                  ECL      0.136060  0.230313  0.153087  0.245054  0.170686   
                  Weather  0.159494  0.201830  0.209565  0.246959  0.266066   
MetaDF            ETTm1    0.306602  0.348971  0.352414  0.376268  0.382595   
                  ETTh1    0.365111  0.388897  0.427475  0.421499  0.465918   
                  ECL      0.134708  0.228838  0.153000  0.244973  0.169068   
                  Weather  0.158300  0.200693  0.206637  0.244936  0.262811   

pred_len                                  720                 Avg            
                                mae       mse       mae       mse       mae  
model             data_id                                                    
DF                ETTm1    0.399900  0.450021  0.436546  0.376018  0.391403  
                  ETTh1    0.454420  0.506832  0.485931  0.448900  0.438991  
                  ECL      0.269703  0.217629  0.302696  0.174991  0.265458  
                  Weather  0.288953  0.346121  0.342470  0.245663  0.270328  
MetaDF$^\dagger$  ETTm1    0.401348  0.449830  0.439005  0.374949  0.392197  
                  ETTh1    0.445092  0.494070  0.481172  0.443254  0.435925  
                  ECL      0.262882  0.203432  0.292526  0.165504  0.257619  
                  Weather  0.287446  0.344149  0.341218  0.243724  0.269015  
MetaDF$^\ddagger$ ETTm1    0.399481  0.443464  0.436364  0.372459  0.390844  
                  ETTh1    0.447250  0.492137  0.474908  0.442071  0.433692  
                  ECL      0.263712  0.202966  0.292270  0.165700  0.257837  
                  Weather  0.288803  0.343256  0.339785  0.244595  0.269344  
MetaDF            ETTm1    0.397512  0.441163  0.434478  0.370693  0.389307  
                  ETTh1    0.448515  0.466485  0.466534  0.431247  0.431361  
                  ECL      0.262404  0.202346  0.291418  0.164780  0.256908  
                  Weather  0.286290  0.342250  0.339154  0.242499  0.267768

In [44]:
show_columns = ['model', 'pred_len', 'data_id', 'mse', 'mae']
show_columns_short = ['model', 'mse', 'mae']
aba_show = pd.concat([df1[show_columns], df2[show_columns_short], df3[show_columns_short], df4[show_columns_short]], axis=1)
# aba_show = pd.concat([df1[show_columns], df2[show_columns], df3[show_columns], df4[show_columns]], axis=1)
aba_show.round(3)

,model,pred_len,data_id,mse,mae,model,mse,mae,model,mse,mae,model,mse,mae
0,DF,96,ETTm1,0.310,0.352,MetaDF$^\dagger$,0.309,0.351,MetaDF$^\ddagger$,0.308,0.351,MetaDF,0.307,0.349
1,DF,192,ETTm1,0.356,0.377,MetaDF$^\dagger$,0.354,0.378,MetaDF$^\ddagger$,0.353,0.377,MetaDF,0.352,0.376
2,DF,336,ETTm1,0.388,0.400,MetaDF$^\dagger$,0.387,0.401,MetaDF$^\ddagger$,0.385,0.399,MetaDF,0.383,0.398
3,DF,720,ETTm1,0.450,0.437,MetaDF$^\dagger$,0.450,0.439,MetaDF$^\ddagger$,0.443,0.436,MetaDF,0.441,0.434
4,DF,Avg,ETTm1,0.376,0.391,MetaDF$^\dagger$,0.375,0.392,MetaDF$^\ddagger$,0.372,0.391,MetaDF,0.371,0.389
5,DF,96,ETTm2,0.175,0.256,MetaDF$^\dagger$,0.171,0.254,MetaDF$^\ddagger$,0.171,0.253,MetaDF,0.170,0.253
6,DF,192,ETTm2,0.243,0.300,MetaDF$^\dagger$,0.235,0.295,MetaDF$^\ddagger$,0.235,0.295,MetaDF,0.234,0.294
7,DF,336,ETTm2,0.297,0.336,MetaDF$^\dagger$,0.292,0.334,MetaDF$^\ddagger$,0.291,0.332,MetaDF,0.290,0.331
8,DF,720,ETTm2,0.394,0.393,MetaDF$^\dagger$,0.391,0.392,MetaDF$^\ddagger$,0.387,0.389,MetaDF,0.387,0.389
9,DF,Avg,ETTm2,0.277,0.321,MetaDF$^\dagger$,0.273,0.319,MetaDF$^\ddagger$,0.271,0.317,MetaDF,0.270,0.317


# write to latex table

In [29]:
import pandas as pd
import numpy as np

stats_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
ablation = pd.read_csv(f'{stats_root}/ablation.csv')

contents = []

for model in ['DF', r'MetaDF$^\dagger$', r'MetaDF$^\ddagger$', 'MetaDF']:
    line = r'\multirow{4}{*}{%s} & ' % model
    if model == 'DF':
        line += r'\multirow{4}{*}{\XSolidBrush} & \multirow{4}{*}{\XSolidBrush}'
    elif model == r'MetaDF$^\dagger$':
        line += r'\multirow{4}{*}{\Checkmark} & \multirow{4}{*}{\XSolidBrush}'
    elif model == r'MetaDF$^\ddagger$':
        line += r'\multirow{4}{*}{\XSolidBrush} & \multirow{4}{*}{\Checkmark}'
    elif model == 'MetaDF':
        line += r'\multirow{4}{*}{\Checkmark} & \multirow{4}{*}{\Checkmark}'
    contents.append(line)
    for i, data_id in enumerate(['ETTm1', 'ETTh1', 'ECL', 'Weather']):
        if i == 0:
            line = '&   ' + data_id + ' && '
        else:
            line = '&&& ' + data_id + ' && '

        _df = ablation[(ablation['model'] == model) & (ablation['data_id'] == data_id)]
        for j, row in enumerate(_df.itertuples()):
            line += r'%.3f & %.3f' % (row.mse, row.mae)
            if j != 4:
                line += ' && '
        line += r' \\'
        contents.append(line)
    contents.append(r'\midrule' + '\n')

print('\n'.join(contents))

\multirow{4}{*}{DF} & \multirow{4}{*}{\XSolidBrush} & \multirow{4}{*}{\XSolidBrush}
&   ETTm1 && 0.310 & 0.352 && 0.356 & 0.377 && 0.388 & 0.400 && 0.450 & 0.437 && 0.376 & 0.391 \\
&&& ETTh1 && 0.372 & 0.391 && 0.430 & 0.424 && 0.486 & 0.454 && 0.507 & 0.486 && 0.449 & 0.439 \\
&&& ECL && 0.143 & 0.237 && 0.161 & 0.252 && 0.178 & 0.270 && 0.218 & 0.303 && 0.175 & 0.265 \\
&&& Weather && 0.160 & 0.203 && 0.210 & 0.247 && 0.267 & 0.289 && 0.346 & 0.342 && 0.246 & 0.270 \\
\midrule

\multirow{4}{*}{MetaDF$^\dagger$} & \multirow{4}{*}{\Checkmark} & \multirow{4}{*}{\XSolidBrush}
&   ETTm1 && 0.309 & 0.351 && 0.354 & 0.378 && 0.387 & 0.401 && 0.450 & 0.439 && 0.375 & 0.392 \\
&&& ETTh1 && 0.372 & 0.394 && 0.432 & 0.424 && 0.475 & 0.445 && 0.494 & 0.481 && 0.443 & 0.436 \\
&&& ECL && 0.135 & 0.230 && 0.154 & 0.246 && 0.170 & 0.263 && 0.203 & 0.293 && 0.166 & 0.258 \\
&&& Weather && 0.159 & 0.202 && 0.208 & 0.246 && 0.265 & 0.287 && 0.344 & 0.341 && 0.244 & 0.269 \\
\midrule

\multirow{4}{*}{

In [45]:
import pandas as pd
import numpy as np

stats_root = '/mnt/tidalfs-bdsz01/usr/panlicheng/workspace/ProjDF-Meta/stats_ML3'
ablation = pd.read_csv(f'{stats_root}/ablation.csv')

# 用字典存储， key = (pred_len, data_id)
min_mse = {}      # (pred_len, data_id) -> (model, value)
second_mse = {}
min_mae = {}
second_mae = {}

grp = ablation.groupby(['pred_len', 'data_id'])
for (plen, d_id), sub in grp:
    # ---------- MSE ----------
    # 按 mse 升序排列，取前两行（可能只有一行）
    mse_sorted = sub.nsmallest(2, 'mse')
    # 第 1 小
    min_mse[(plen, d_id)] = (mse_sorted.iloc[0]['model'],
                             mse_sorted.iloc[0]['mse'])
    # 第 2 小（如果真的存在第二行）
    if len(mse_sorted) > 1:
        second_mse[(plen, d_id)] = (mse_sorted.iloc[1]['model'],
                                    mse_sorted.iloc[1]['mse'])
    else:
        second_mse[(plen, d_id)] = (None, np.inf)

    # ---------- MAE ----------
    mae_sorted = sub.nsmallest(2, 'mae')
    min_mae[(plen, d_id)] = (mae_sorted.iloc[0]['model'],
                             mae_sorted.iloc[0]['mae'])
    if len(mae_sorted) > 1:
        second_mae[(plen, d_id)] = (mae_sorted.iloc[1]['model'],
                                    mae_sorted.iloc[1]['mae'])
    else:
        second_mae[(plen, d_id)] = (None, np.inf)


def wrap_val(val, val_type, plen, d_id):
    if val_type == 'mse':
        min_model, min_val = min_mse[(plen, d_id)]
        sec_model, sec_val = second_mse[(plen, d_id)]
    else:  # mae
        min_model, min_val = min_mae[(plen, d_id)]
        sec_model, sec_val = second_mae[(plen, d_id)]

    if np.isclose(val, min_val):
        return r'\bst{%.3f}' % val
    elif np.isclose(val, sec_val):
        return r'\subbst{%.3f}' % val
    else:
        return r'%.3f' % val

contents = []

for model in ['DF', r'MetaDF$^\dagger$', r'MetaDF$^\ddagger$', 'MetaDF']:
    # ---- 第一列（模型名）以及两列 check / cross ----
    line = r'\multirow{4}{*}{%s} & ' % model
    if model == 'DF':
        line += r'\multirow{4}{*}{\XSolidBrush} & \multirow{4}{*}{\XSolidBrush}'
    elif model == r'MetaDF$^\dagger$':
        line += r'\multirow{4}{*}{\Checkmark} & \multirow{4}{*}{\XSolidBrush}'
    elif model == r'MetaDF$^\ddagger$':
        line += r'\multirow{4}{*}{\XSolidBrush} & \multirow{4}{*}{\Checkmark}'
    elif model == 'MetaDF':
        line += r'\multirow{4}{*}{\Checkmark} & \multirow{4}{*}{\Checkmark}'
    contents.append(line)

    # ---- 对四个 data_id（ETTh1、ETTm1、ECL、Weather）循环 ----
    for i, data_id in enumerate(['ETTm1', 'ETTh1', 'ECL', 'Weather']):
        # 开头的 “&   ETTh1 && ” 之类的固定格式
        if i == 0:
            line = '&   ' + data_id + ' && '
        else:
            line = '&&& ' + data_id + ' && '

        # 取出该 (model, data_id) 的所有 pred_len 行（5 行：96,192,336,720,Avg）
        _df = ablation[(ablation['model'] == model) &
                           (ablation['data_id'] == data_id)]

        # 按原始顺序（pred_len=96 → 192 → 336 → 720 → Avg）遍历
        for j, row in enumerate(_df.itertuples()):
            # 对 mse 与 mae 分别调用包装函数
            mse_str = wrap_val(row.mse, 'mse', row.pred_len, row.data_id)
            mae_str = wrap_val(row.mae, 'mae', row.pred_len, row.data_id)

            line += f'{mse_str} & {mae_str}'
            if j != 4:          # 不是最后一列，需要再加上 “&&”
                line += ' && '
        line += r' \\'
        contents.append(line)

    if model != 'MetaDF':
        contents.append(r'\midrule' + '\n')

print('\n'.join(contents))


\multirow{4}{*}{DF} & \multirow{4}{*}{\XSolidBrush} & \multirow{4}{*}{\XSolidBrush}
&   ETTm1 && 0.310 & 0.352 && 0.356 & 0.377 && 0.388 & 0.400 && 0.450 & 0.437 && 0.376 & 0.391 \\
&&& ETTh1 && 0.372 & 0.391 && 0.430 & 0.424 && 0.486 & 0.454 && 0.507 & 0.486 && 0.449 & 0.439 \\
&&& ECL && 0.143 & 0.237 && 0.161 & 0.252 && 0.178 & 0.270 && 0.218 & 0.303 && 0.175 & 0.265 \\
&&& Weather && 0.160 & 0.203 && 0.210 & 0.247 && 0.267 & 0.289 && 0.346 & 0.342 && 0.246 & 0.270 \\
\midrule

\multirow{4}{*}{MetaDF$^\dagger$} & \multirow{4}{*}{\Checkmark} & \multirow{4}{*}{\XSolidBrush}
&   ETTm1 && 0.309 & 0.351 && 0.354 & 0.378 && 0.387 & 0.401 && 0.450 & 0.439 && 0.375 & 0.392 \\
&&& ETTh1 && 0.372 & 0.394 && 0.432 & 0.424 && \subbst{0.475} & \bst{0.445} && 0.494 & 0.481 && 0.443 & 0.436 \\
&&& ECL && \subbst{0.135} & \subbst{0.230} && 0.154 & 0.246 && \subbst{0.170} & \subbst{0.263} && 0.203 & 0.293 && \subbst{0.166} & \subbst{0.258} \\
&&& Weather && \subbst{0.159} & \subbst{0.202} && \subbst